# Project 3: Hospital Patient Queue System

**Course:** Data Structures and Algorithms  
**Group:** Group 5  

## Group members

| # | Member name | Student ID |
|---|---|---|
| 1 | Simon Wario | INTE/M/0903/05/25 |

## Problem statement

A hospital needs a simple way to track patients waiting to see a doctor. The system stores each patient's ID, name, age, and reason for visiting. Patients are served in their arrival order: **First In, First Out (FIFO)**.

The system must add and call patients, display and search waiting patients, remove a patient by ID, count waiting patients, find the oldest patient, and show the first patient without removing them.

## Step 1: Choose the data structure

A queue represents a waiting line. New patients join at the **rear** and the next patient to be seen leaves from the **front**. This implementation uses a singly linked list with `front` and `rear` references. Unlike removing the first item of a Python list, dequeue does not shift the remaining patients.

## Step 2: Explain the algorithms

- **Enqueue:** create a node and connect it after the rear. If the queue is empty, the new node becomes both front and rear.
- **Dequeue / call next:** save the front patient's record and advance front to the next node. If the queue becomes empty, also set rear to `None`.
- **Traversal:** start at front and follow `next` until the end. Display, search, removal, and finding the oldest patient use traversal.
- **Linear search:** compare each patient's ID with the requested ID, returning the matching record or `None`.
- **Removal by ID:** track the current and previous nodes, bypass the match, and update front or rear when an endpoint is removed.

The queue maintains these invariants: `front` and `rear` are both `None` exactly when it is empty; otherwise rear is the last reachable node and `rear.next` is `None`. The `size` counter equals the number of reachable nodes.

In [1]:
from dataclasses import dataclass


@dataclass
class Patient:
    patient_id: str
    name: str
    age: int
    reason: str

    def __post_init__(self):
        if not isinstance(self.patient_id, str) or not self.patient_id.strip():
            raise ValueError("Patient ID must be a non-empty string.")
        if not isinstance(self.name, str) or not self.name.strip():
            raise ValueError("Patient name must be a non-empty string.")
        if type(self.age) is not int or self.age < 0:
            raise ValueError("Patient age must be a non-negative integer.")
        if not isinstance(self.reason, str) or not self.reason.strip():
            raise ValueError("Reason for visit must be a non-empty string.")

    def __str__(self):
        return (
            f"ID: {self.patient_id} | Name: {self.name} | "
            f"Age: {self.age} | Reason: {self.reason}"
        )


class QueueNode:
    def __init__(self, patient):
        self.patient = patient
        self.next = None


class HospitalPatientQueue:
    def __init__(self):
        self.front = None
        self.rear = None
        self.size = 0

    def is_empty(self):
        return self.front is None

    def enqueue(self, patient):
        if not isinstance(patient, Patient):
            raise TypeError("Only Patient records can be added to the queue.")
        if self.search_patient(patient.patient_id) is not None:
            raise ValueError(f"Patient ID {patient.patient_id!r} is already waiting.")

        node = QueueNode(patient)
        if self.is_empty():
            self.front = self.rear = node
        else:
            self.rear.next = node
            self.rear = node
        self.size += 1
        return patient

    def dequeue(self):
        if self.is_empty():
            return None
        patient = self.front.patient
        self.front = self.front.next
        self.size -= 1
        if self.front is None:
            self.rear = None
        return patient

    def peek(self):
        return None if self.is_empty() else self.front.patient

    def search_patient(self, patient_id):
        current = self.front
        while current is not None:
            if current.patient.patient_id.casefold() == str(patient_id).casefold():
                return current.patient
            current = current.next
        return None

    def remove_patient(self, patient_id):
        previous = None
        current = self.front
        while current is not None:
            if current.patient.patient_id.casefold() == str(patient_id).casefold():
                if previous is None:
                    self.front = current.next
                else:
                    previous.next = current.next
                if current is self.rear:
                    self.rear = previous
                self.size -= 1
                if self.front is None:
                    self.rear = None
                return current.patient
            previous = current
            current = current.next
        return None

    def count_patients(self):
        return self.size

    def find_oldest_patient(self):
        if self.is_empty():
            return None
        oldest = self.front.patient
        current = self.front.next
        while current is not None:
            if current.patient.age > oldest.age:
                oldest = current.patient
            current = current.next
        return oldest

    def display_patients(self):
        if self.is_empty():
            print("No patients are waiting.")
            return
        current = self.front
        print("Waiting patients (front to rear):")
        while current is not None:
            print(f"  {current.patient}")
            current = current.next


## Step 3: Add sample data and demonstrate the required operations

The sample records below are fictional. The demonstration displays the queue, peeks at the first patient, searches by ID, identifies the oldest patient, counts the queue, removes a waiting patient, and calls the next patient in FIFO order.

In [2]:
queue = HospitalPatientQueue()
sample_patients = [
    Patient("P101", "Mary Otieno", 22, "Fever"),
    Patient("P102", "John Kamau", 35, "Routine checkup"),
    Patient("P103", "Jane Wanjiku", 18, "Allergy"),
    Patient("P104", "Peter Mwangi", 35, "Headache"),
]

for patient in sample_patients:
    queue.enqueue(patient)

print("1. Display the waiting queue")
queue.display_patients()
print("\n2. First patient (peek):", queue.peek())
print("\n3. Search for P103:", queue.search_patient("P103"))
print("4. Search for missing ID:", queue.search_patient("P999"))
print("\n5. Number waiting:", queue.count_patients())
print("6. Oldest patient:", queue.find_oldest_patient())
print("\n7. Remove P102:", queue.remove_patient("P102"))
print("8. Call next patient (dequeue):", queue.dequeue())
print("\n9. Queue after operations:")
queue.display_patients()
print("Patients still waiting:", queue.count_patients())

1. Display the waiting queue
Waiting patients (front to rear):
  ID: P101 | Name: Mary Otieno | Age: 22 | Reason: Fever
  ID: P102 | Name: John Kamau | Age: 35 | Reason: Routine checkup
  ID: P103 | Name: Jane Wanjiku | Age: 18 | Reason: Allergy
  ID: P104 | Name: Peter Mwangi | Age: 35 | Reason: Headache

2. First patient (peek): ID: P101 | Name: Mary Otieno | Age: 22 | Reason: Fever

3. Search for P103: ID: P103 | Name: Jane Wanjiku | Age: 18 | Reason: Allergy
4. Search for missing ID: None

5. Number waiting: 4
6. Oldest patient: ID: P102 | Name: John Kamau | Age: 35 | Reason: Routine checkup

7. Remove P102: ID: P102 | Name: John Kamau | Age: 35 | Reason: Routine checkup
8. Call next patient (dequeue): ID: P101 | Name: Mary Otieno | Age: 22 | Reason: Fever

9. Queue after operations:
Waiting patients (front to rear):
  ID: P103 | Name: Jane Wanjiku | Age: 18 | Reason: Allergy
  ID: P104 | Name: Peter Mwangi | Age: 35 | Reason: Headache
Patients still waiting: 2


## Step 4: Test normal cases and edge cases

These assertions check empty-queue behavior, FIFO order, case-insensitive ID search, duplicate and invalid input handling, removal from the front/middle/rear, the single-node case, size and rear consistency, and oldest-patient tie behavior. A successful run prints a PASS message.

In [3]:
def run_tests():
    empty = HospitalPatientQueue()
    assert empty.is_empty()
    assert empty.dequeue() is None
    assert empty.peek() is None
    assert empty.search_patient("missing") is None
    assert empty.remove_patient("missing") is None
    assert empty.find_oldest_patient() is None
    assert empty.count_patients() == 0

    q = HospitalPatientQueue()
    a = Patient("A", "Ari", 40, "Cough")
    b = Patient("B", "Bo", 60, "Checkup")
    c = Patient("C", "Cy", 60, "Fever")
    d = Patient("D", "Dee", 20, "Injury")
    for patient in (a, b, c):
        q.enqueue(patient)
    assert q.count_patients() == 3
    assert q.peek() is a
    assert q.dequeue() is a
    assert q.search_patient("b") is b
    assert q.search_patient("unknown") is None
    assert q.find_oldest_patient() is b  # First patient wins an age tie.

    try:
        q.enqueue(Patient("B", "Duplicate", 25, "Checkup"))
    except ValueError:
        pass
    else:
        raise AssertionError("Duplicate patient IDs must be rejected.")

    # Remove from the middle, then verify front and rear removal.
    q.enqueue(d)
    assert q.remove_patient("C") is c
    assert q.front.patient is b and q.rear.patient is d
    assert q.remove_patient("B") is b
    assert q.front.patient is d and q.rear.patient is d
    assert q.remove_patient("D") is d
    assert q.is_empty() and q.front is None and q.rear is None

    # Removing the sole remaining node resets both endpoints.
    q.enqueue(c)
    assert q.remove_patient("C") is c
    assert q.is_empty() and q.front is None and q.rear is None
    assert q.count_patients() == 0

    # FIFO dequeue also resets rear after the final node leaves.
    q.enqueue(a)
    q.enqueue(b)
    assert q.remove_patient("A") is a
    assert q.front.patient is b and q.rear.patient is b
    assert q.dequeue() is b
    assert q.is_empty() and q.rear is None

    invalid_patients = [
        ("", "Name", 1, "Reason"),
        ("ID", "", 1, "Reason"),
        ("ID", "Name", -1, "Reason"),
        ("ID", "Name", True, "Reason"),
        ("ID", "Name", 1, " "),
    ]
    for args in invalid_patients:
        try:
            Patient(*args)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Invalid patient data was accepted: {args!r}")

    print("All Hospital Patient Queue tests passed.")


run_tests()

All Hospital Patient Queue tests passed.


## Step 5: Analyse time and space complexity

Let **n** be the number of patients waiting.

| Operation | Time | Reason |
|---|---:|---|
| `enqueue` | O(n) | Duplicate-ID checking traverses the queue; linking the new rear node itself is O(1). |
| `dequeue` / call next | O(1) | Advance the front reference. |
| `peek`, `is_empty`, `count_patients` | O(1) | Read a stored reference or size counter. |
| `display_patients` | O(n) | Visit every waiting patient. |
| `search_patient` | O(n) | Linear search may inspect every node. |
| `remove_patient` | O(n) | Search for the ID; unlinking the found node is O(1). |
| `find_oldest_patient` | O(n) | Compare every patient's age. |
| Space | O(n) | One linked node is stored per waiting patient. |

Because duplicate IDs are rejected by searching first, enqueue is O(n) overall in this version. Without that additional check, adding to the rear is O(1). For a real hospital deployment, the queue would need persistence, authentication, privacy safeguards, and a clinically appropriate triage policy; FIFO alone does not determine medical priority.

## Conclusion

This project models a hospital waiting line with a linked FIFO queue. It implements every required operation, demonstrates traversal and linear search, and tests queue endpoint updates and empty/single-patient edge cases. The linked implementation makes calling the next patient efficient while keeping the queue's order explicit.
